In [1]:
from ash import *

ashpath: /Users/tom/Documents/ucl/projects/ash-fork/ash
Sys path: ['/Users/tom/Documents/ucl/projects/ash-fork/ash', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python311.zip', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/lib-dynload', '', '/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages', '__editable__.ash-0.95.finder.__path_hook__']
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                                           ASH                                            
                              A MULTISCALE MODELLING PROGRAM                              
                                        Version: 0.9dev                                        
                               Git c

In [2]:
BASIS = 'sto-3g'
XC = 'b3lyp'
CHARGE = 0
MULT = 1

In [3]:
#Defining fragment
frag = Fragment(xyzfile="system_aftersolvent.xyz", charge=CHARGE, mult=MULT)


--------------------------------------------------------------------------------
                                New ASH fragment                                
--------------------------------------------------------------------------------

ASH Fragment creation
Reading coordinates from XYZ file 'system_aftersolvent.xyz' into fragment.
Creating/Updating fragment attributes...
Number of Atoms in fragment: 2637
Formula: O877H1758C2
Label: system_aftersolvent
Charge: 0 Mult: 1

--------------------------------------------------------------------------------


In [4]:
xyz_list = [i for i in zip(frag.elems, frag.coords)]

lines = [str(len(xyz_list)), '']
for symbol, coords in xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

xyz_string = "\n".join(lines)

In [5]:
qm_atoms = list(range(9))

In [6]:
qm_atoms

[0, 1, 2, 3, 4, 5, 6, 7, 8]

In [7]:
qm_xyz_list = [
    (frag.elems[i], frag.coords[i])
    for i in qm_atoms
]

lines = [str(len(qm_xyz_list)), '']
for symbol, coords in qm_xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

qm_xyz_string = "\n".join(lines)
print(qm_xyz_string)

9

C 0.0 0.0 0.0
C -1.187 0.984 0.0
O -2.406 0.224 0.0
H -3.117 0.913 0.0
H -1.125 1.631 0.883
H -1.125 1.631 -0.883
H 0.939 0.546 0.0
H -0.036 -0.633 0.881
H -0.036 -0.633 -0.881


In [8]:
N_ACT = 2

In [9]:
nbed_theory = NbedTheory(
    geometry=qm_xyz_string,
    n_active_atoms=N_ACT,
    basis=BASIS,
    xc_functional=XC,
    projector='mu',
    localization='spade',
    # run_ccsd_emb=True
)



                     #####################################                      
                     #                                   #                      
                     #     NbedTheory initialization     #                      
                     #                                   #                      
                     #####################################                      


In [10]:
# water_xml = "/opt/homebrew/Caskroom/miniconda/base/envs/ash-conda/lib/python3.11/site-packages/openmm/app/data/amber14/tip3p.xml"
water_xml = "amber14/tip3p.xml"

frozen_atoms=listdiff(frag.allatoms,qm_atoms)

openmm_theory = OpenMMTheory(
    xmlfiles=["openff_LIG.xml", water_xml], 
    pdbfile="system_aftersolvent.pdb", 
    # periodic=True, 
    # autoconstraints=None,
    # rigidwater=False,
    # frozen_atoms=qm_atoms,
)



                           #########################                            
                           #                       #                            
                           #     OpenMM Theory     #                            
                           #                       #                            
                           #########################                            
OpenMM CPU threads set to: 1
Imported OpenMM library version: 8.3.1

--------------------------------------------------------------------------------
                             Defining OpenMM object                             
--------------------------------------------------------------------------------

Printlevel: 2
HBonds option: X-H bond lengths will automatically be constrained
AutoConstraint setting: HBonds
Rigidwater constraints: True
Hydrogenmass option: 1.5 Da
Using platform: CPU

--------------------------------------------------------------------------------
          

In [11]:
qmmm_theory = QMMMTheory(
    qm_theory = nbed_theory,
    mm_theory = openmm_theory,
    fragment = frag,
    qm_charge = CHARGE,
    qm_mult = MULT,
    qmatoms = qm_atoms,
    printlevel = 3,
)



                            ########################                            
                            #                      #                            
                            #     QM/MM Theory     #                            
                            #                      #                            
                            ########################                            
QM-theory: NbedTheory
MM-theory: OpenMMTheory
All atoms in fragment: 2637
QM region (9 atoms): [0, 1, 2, 3, 4, 5, 6, 7, 8]
MM region (2628 atoms)
QM/MM object selected to use 1 cores
Embedding: elstat
No atomcharges list passed to QMMMTheory object
Getting system charges from OpenMM object
QM-region coordinates (before linkatoms):
   0    C   0.00000000    0.00000000    0.00000000
   1    C  -1.18700000    0.98400000    0.00000000
   2    O  -2.40600000    0.22400000    0.00000000
   3    H  -3.11700000    0.91300000    0.00000000
   4    H  -1.12500000    1.63100000    0.88300000
   5 

In [12]:
qmmm_theory

In [13]:
MolecularDynamics(
    fragment=frag,
    theory=qmmm_theory,
    timestep=0.001,
    simulation_steps=120,
    traj_frequency=1,
    temperature=300,
    # integrator='LangevinIntegrator',
    coupling_frequency=1,
    charge=CHARGE,
    mult=MULT
)



                     ######################################                     
                     #                                    #                     
                     #     OpenMM MD wrapper function     #                     
                     #                                    #                     
                     ######################################                     


              ####################################################              
              #                                                  #              
              #     OpenMM Molecular Dynamics Initialization     #              
              #                                                  #              
              ####################################################              
Analyzing theory input to OpenMM_MDclass
This is an QMMMTheory object
Turning on externalforce option.
Added force
System is non-periodic. Setting enforcePeriodicBox to False

----------

/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/pyscf/dft/libxc.py:511: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -153.02650021305  <S^2> = 6.77014e-09  2S+1 = 1

WARN: Incompatible dm dimension. Treat dm as RHF density matrix.



/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/occupied/spade.py:126: RuntimeWarning: divide by zero encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/occupied/spade.py:126: RuntimeWarning: overflow encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/occupied/spade.py:126: RuntimeWarning: invalid value encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/occupied/spade.py:154: RuntimeWarning: divide by zero encountered in matmul
  c_loc_occ = occupied_orbitals @ right_vectors.T
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/sit

converged SCF energy = -39.7149622806322  <S^2> = 7.0268984e-08  2S+1 = 1.0000001


/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/driver.py:431: RuntimeWarning: divide by zero encountered in matmul
  env_projector_alpha = s_mat @ self.localized_system.dm_enviro @ s_mat
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/driver.py:431: RuntimeWarning: overflow encountered in matmul
  env_projector_alpha = s_mat @ self.localized_system.dm_enviro @ s_mat
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/driver.py:431: RuntimeWarning: invalid value encountered in matmul
  env_projector_alpha = s_mat @ self.localized_system.dm_enviro @ s_mat
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/driver.py:437: RuntimeWarning: divide by zero encountered in matmul
  env_projector_beta = s_mat @ self.localized_system.beta_dm_enviro @ s_mat
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/driver.py:437: RuntimeWarnin

MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

KeyboardInterrupt: 

In [ ]:
# Optimizer(
#     fragment=frag, 
#     theory=qmmm_theory, 
#     ActiveRegion=True, 
#     actatoms=qm_atoms
# )